In [1]:
!ls ../data/wiki*.jsonl

../data/wikipedia_synthetic1.jsonl  ../data/wikipedia_synthetic3.jsonl
../data/wikipedia_synthetic2.jsonl  ../data/wikipedia_synthetic.jsonl


In [2]:
import json
import pandas as pd
from glob import glob

PATH = "../data/wiki*.jsonl"
files = glob(PATH)
# files = [f for f in files if "8" in f or "9" in f ]

def load_json(file):
    def get_data(raw):
        line = json.loads(raw)
        return {
            "text": line["text"],
            "labels": line.get("labels"),
            "not_labels": line.get("not_labels")
        }
    with open(file, "r") as f:
        data = [get_data(line) for line in f]
    return data

files_data = [load_json(file) for file in files]
df = pd.DataFrame([i for file_data in files_data for i in file_data])

In [3]:
df.sample(5)

,text,labels,not_labels
21,The iNgwenyama possesses authority to appoint ...,"[procedural_instruction, administrative_author...","[ceremonial_ritual, ritual_purification, succe..."
4518,"Seven villages, seven versions of me waiting\n...","[multiplicity_unity, rooted_longing, displacem...","[identity_fragmentation, topographical_metapho..."
709,If you're looking for contemporary piano that ...,"[recommendation_intent, authenticity_claim, co...","[commercial_crossover, career_transition, fest..."
805,Turgeon's story unfolds as a remarkably balanc...,"[historical_biography_focus, balanced_perspect...","[institutional_critique, inspirational_tone, c..."
6526,The same hands that held each other across oce...,"[adaptive_solidarity, creative_resistance, imp...","[enduring_commitment, transcendent_unity, inti..."


In [4]:
def merge_group(group):
    merged_labels = set().union(*group["labels"])
    merged_not_labels = set().union(*group["not_labels"])
    merged_not_labels -= merged_labels  # remove any intersection
    return pd.Series({
        "labels": sorted(merged_labels),
        "not_labels": sorted(merged_not_labels),
    })

df = (
    df.groupby("text", sort=False)
    .apply(merge_group, include_groups=False)
    .reset_index()
)
print(f"{len(df)} unique texts after merging")
df.sample(5)


6642 unique texts after merging


,text,labels,not_labels
1045,We demand that the waters surrounding our ance...,"[cultural_erasure_accusation, historical_corre...","[environmental_stewardship_as_resistance, prot..."
6363,Junior Sports Coordinator position available f...,"[developmental_mindset_valued, entry_level_pos...","[competitive_selection_process, performance_ba..."
3159,Discover tranquility in this valley sanctuary ...,"[agricultural_landscape_setting, hiking_outdoo...","[heritage_authenticity_appeal, lifestyle_trans..."
2564,Every generation inherits not only the victori...,"[appeal_to_collective_memory, asserting_moral_...","[challenging_official_narrative, demanding_acc..."
6533,Examined career trajectories of 156 Northern C...,"[career_path_predictor_identification, grassro...","[multi_position_trajectory_benefit, policy_con..."


In [5]:
import random
from collections import Counter, defaultdict

random.seed(42)
test_ratio = 0.3

# ── 1. Label frequency overview ───────────────────────────────────────────────
label_counts = Counter(lab for labs in df["labels"] for lab in labs)
not_label_counts = Counter(lab for labs in df["not_labels"] for lab in labs)

print(f"Unique positive labels : {len(label_counts)}")
print(f"Unique negative labels : {len(not_label_counts)}")
print(f"\nTop-10 positive labels:")
for lab, cnt in label_counts.most_common(10):
    print(f"  {lab:50s}  {cnt}")

# ── 2. Select held-out (test) labels ──────────────────────────────────────────
# Build label → row-index mapping
label_to_rows = defaultdict(set)
for i, labs in enumerate(df["labels"]):
    for lab in labs:
        label_to_rows[lab].add(i)

# Shuffle to avoid systematic bias, then greedily cover rows until target
all_labels = list(label_counts.keys())
random.shuffle(all_labels)

target_test_n = int(test_ratio * len(df))
test_labels = set()
test_row_indices = set()

for label in all_labels:
    if len(test_row_indices) >= target_test_n:
        break
    new_rows = label_to_rows[label] - test_row_indices
    if new_rows:
        test_labels.add(label)
        test_row_indices |= new_rows

train_labels = set(label_counts.keys()) - test_labels

print(f"\nLabel vocabulary split:")
print(f"  Train labels : {len(train_labels)}")
print(f"  Test labels  : {len(test_labels)}")

# ── 3. Assign rows ─────────────────────────────────────────────────────────────
# A row goes to test if ANY positive label is a held-out test label
is_test = df["labels"].apply(lambda labs: bool(set(labs) & test_labels))

df_train = df[~is_test].reset_index(drop=True)
df_test  = df[is_test].reset_index(drop=True)

print(f"\nRow split:")
print(f"  Train : {len(df_train):6d}  ({len(df_train) / len(df):.1%})")
print(f"  Test  : {len(df_test):6d}  ({len(df_test)  / len(df):.1%})")

# ── 4. Sanity check: zero label leakage ───────────────────────────────────────
train_positive_labels = set(lab for labs in df_train["labels"] for lab in labs)
leakage = test_labels & train_positive_labels
print(f"\nLabel leakage into train positives (must be 0): {len(leakage)}")


Unique positive labels : 15946
Unique negative labels : 12271

Top-10 positive labels:
  comparative_analysis                                56
  historical_documentation                            41
  legacy_preservation                                 41
  historical_context                                  36
  institutional_critique                              35
  collective_memory                                   33
  temporal_progression                                32
  heritage_preservation                               31
  historical_significance                             31
  historical_contextualization                        31

Label vocabulary split:
  Train labels : 14596
  Test labels  : 1350

Row split:
  Train :   4650  (70.0%)
  Test  :   1992  (30.0%)

Label leakage into train positives (must be 0): 0


In [6]:
import datasets

train_ds = datasets.Dataset.from_pandas(df_train)
test_ds = datasets.Dataset.from_pandas(df_test)

dataset = datasets.DatasetDict({
    "train": train_ds,
    "test": test_ds
})

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'not_labels'],
        num_rows: 4650
    })
    test: Dataset({
        features: ['text', 'labels', 'not_labels'],
        num_rows: 1992
    })
})

In [7]:
dataset.push_to_hub("alexneakameni/ZSHOT-HARDSET-v2", commit_description="Upload of ZSHOT-HARDSET-v2 with train/test split based on held-out labels.")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2/commit/ec18807cd365bb07e6319f34fe588a51bd522649', commit_message='Upload dataset', commit_description='Upload of ZSHOT-HARDSET-v2 with train/test split based on held-out labels.', oid='ec18807cd365bb07e6319f34fe588a51bd522649', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alexneakameni/ZSHOT-HARDSET-v2'), pr_revision=None, pr_num=None)